In [2]:
# load dungeon data

import json
import random
from pathlib import Path

data_path = Path("../datasets/dungeon_10k_4_8_3_5_mkr.jsonl")
with open(data_path) as f:
    data = [json.loads(line) for line in f]

# strip _id fields
for entry in data:
    if "_id" in entry:
        del entry["_id"]

# Split into train/eval sets
random.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
eval_data = data[split_idx:]

print(f"Train set: {len(train_data)} records")
print(f"Eval set: {len(eval_data)} records")

# print first data point
print(json.dumps(data[0], indent=2))

Train set: 8000 records
Eval set: 2000 records
{
  "door": 4,
  "key_color": "blue",
  "corridor": [
    {
      "door_no": 6,
      "green_key": "diamonds",
      "red_key": "diamonds",
      "blue_key": "diamonds"
    },
    {
      "monsters": [
        "troll",
        "goblin"
      ],
      "door_no": 3,
      "blue_key": "spellbooks",
      "red_key": "gemstones",
      "green_key": "gemstones"
    },
    {
      "monsters": [
        "orc"
      ],
      "door_no": 7,
      "green_key": "artifacts",
      "blue_key": "artifacts",
      "red_key": "gold"
    },
    {
      "door_no": 1,
      "blue_key": "gemstones",
      "green_key": "diamonds",
      "red_key": "artifacts"
    },
    {
      "monsters": [
        "dragon"
      ],
      "door_no": 5,
      "red_key": "spellbooks",
      "green_key": "artifacts",
      "blue_key": "spellbooks"
    },
    {
      "monsters": [
        "dragon"
      ],
      "door_no": 0,
      "red_key": "artifacts",
      "green_key": "diamon

In [ ]:
from origami import ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.training import TableLogCallback, accuracy

config = OrigamiConfig(
    model=ModelConfig(
        d_model=192,
        n_heads=8,
        n_layers=6,
        d_ff=784,
        dropout=0.0,
        use_grammar_constraints=True,
    ),
    training=TrainingConfig(
        shuffle_keys=False,
        upscale_factor=1,
        batch_size=100,
        warmup_steps=1000,
        learning_rate=5e-4,
        eval_strategy="epoch",
        eval_epochs=5,
        eval_metrics={"acc": accuracy},
        eval_sample_size=100,
        target_key="treasure",
    ),
)

pipeline = OrigamiPipeline(config)
pipeline.fit(train_data, eval_data=eval_data, callbacks=[TableLogCallback(print_every=50)], epochs=250, verbose=True)

Vocabulary size: 40
Model parameters: 2,771,080
Training device: mps

Training interrupted at epoch 0, step 14


OrigamiPipeline(numeric_mode='none', fitted)

In [ ]:
pipeline.save("dungeon_pipeline.pt")

In [ ]:
from origami import OrigamiPipeline

pipeline = OrigamiPipeline.load("dungeon_pipeline.pt")

In [ ]:
from origami.training import accuracy

pipeline.evaluate(eval_data, metrics={"acc": accuracy})

In [ ]:
doc = pipeline.generate(1)[0]

print(json.dumps(doc, indent=2))